<a href="https://colab.research.google.com/github/ShouryaWired/TunnelBot/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connect to T4 runtime

In [ ]:
!nvidia-smi

Tue Nov 24 15:20:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


Install's ollama

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os, subprocess, time

!pkill -9 ollama
time.sleep(2)

os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

In [ ]:
# To check Context, Attention, KV Cache set

# !tr '\0' '\n' < /proc/$(pgrep -o ollama)/environ | grep OLLAMA

OLLAMA_CONTEXT_LENGTH=32768
OLLAMA_FLASH_ATTENTION=1
OLLAMA_KV_CACHE_TYPE=f16
OLLAMA_NUM_PARALLEL=1
OLLAMA_KEEP_ALIVE=30m


Test Run

In [ ]:
!ollama run hf.co/vcruz305/Qwen3.8-27B-AEON-ULTIMATE-UNCENSORED-GGUF:Q2_K "Say hi in one sentence."
!ollama ps

In [ ]:
# optional tests

# import requests, json

# r = requests.post("http://localhost:11434/v1/chat/completions", json={
#     "model": "hf.co/vcruz305/Qwen3.8-27B-AEON-ULTIMATE-UNCENSORED-GGUF:Q2_K",
#     "messages": [{"role": "user", "content": "What is the weather in Paris?"}],
#     "tools": [{
#         "type": "function",
#         "function": {
#             "name": "get_weather",
#             "description": "Get weather for a city",
#             "parameters": {
#                 "type": "object",
#                 "properties": {"city": {"type": "string"}},
#                 "required": ["city"],
#             },
#         },
#     }],
# })
# print(json.dumps(r.json()["choices"][0]["message"], indent=2))

{
  "role": "assistant",
  "content": "",
  "reasoning": "We need answer question: \"What is the weather in Paris?\" Need call tool only once. Function name get_weather. Need provide city \"Paris\". No dependencies.",
  "tool_calls": [
    {
      "id": "2hWnumTSW9UaoPRR16bTo1md7kELXKfP",
      "index": 0,
      "type": "function",
      "function": {
        "name": "get_weather",
        "arguments": "{\"city\":\"Paris\"}"
      }
    }
  ]
}


Creating a tunnel form colab to PC.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
subprocess.Popen("./cloudflared tunnel --url http://localhost:11434 --http-host-header localhost:11434 > tunnel.log 2>&1", shell=True)
time.sleep(8)
# !grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | head -1

This is to grab the URL. its the subpart of above snippet (separated for repetitive use).

In [ ]:
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | tail -1

https://compounds-abroad-rely-were.trycloudflare.com


In [ ]:
# "curl URL/v1/models/" for checking connectivity on PC.

This Task is important for Registering the provider on OPENCODE. In case of multiple providers add the ln6 - ln22 to the opencode.json's provider section.

In [ ]:
# ~/.config/opencode/opencode.json

# {
#   "$schema": "https://opencode.ai/config.json",
#   "provider": {
#     "colab": {
#       "npm": "@ai-sdk/openai-compatible",
#       "name": "Colab Ollama",
#       "options": {
#         "baseURL": "https://YOUR-URL.trycloudflare.com/v1"
#       },
#       "models": {
#         "hf.co/vcruz305/Qwen3.8-27B-AEON-ULTIMATE-UNCENSORED-GGUF:Q2_K": {
#           "name": "Qwen3.8 27B Q2 (Colab)",
#           "limit": {
#             "context": 32768,
#             "output": 8192
#           }
#         }
#       }
#     }
#   },
#   "model": "colab/hf.co/vcruz305/Qwen3.8-27B-AEON-ULTIMATE-UNCENSORED-GGUF:Q2_K"
# }

This Stops the Model

In [ ]:
!ollama stop hf.co/vcruz305/Qwen3.8-27B-AEON-ULTIMATE-UNCENSORED-GGUF:Q2_K